# Magnitude equation — conditional OLS

Given an arbitrage is open, *how large* is the round-trip gap? OLS on the log gap size, conditional
on existence (`D == 1`):

$$
\log \mathrm{Gap}_{p,t}=\alpha_p
+\theta_1\,\log(1+\mathrm{Gap}_{p,t-1})+\theta_2\,\log(\text{base\_fee}_t)+\theta_3\,\text{gas\_util}_{t-1}
+\theta_4\,\log(1+\text{tip\_p90}_{t-1})+\theta_5\,\overline{\log(1+\text{mev})}_{p,t-1}
+\theta_6\,\overline{\text{nb\_swaps\_ewma}}_{p,t-1}+\theta_7\,\log(\text{ewma\_vol}_t)
+\theta_8\,\overline{\Delta\log L}_{p,t-1}+\varepsilon_{p,t}\qquad\text{s.t. } D_{p,t}=1
$$

Same panel and lag discipline as the existence equation, but OLS. The lagged gap enters as
`log(1+Gap_{t-1})` since it can be exactly 0 (a freshly-opened spell). All logic lives in
`arblib.estimation`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

from arblib import estimation as est
from arblib.config import STUDY as S

panel = est.load_panel(S)
print("panel:", panel.shape)

panel: (4704, 28)


## Magnitude sample — condition on existence

`build_magnitude_sample` keeps `D == 1` rows, forms `log_gap` / `log1p_gap_lag`, and drops pairs
with too few observations.

In [2]:
mag = est.build_magnitude_sample(panel, quantile=0.2)

q20: dropping 15 pool pairs with no D variation (uninformative under pair FE): ['uniswap_1_vs_uniswap_4', 'uniswap_1_vs_uniswap_5', 'uniswap_2_vs_pancake_2', 'uniswap_2_vs_uniswap_3', 'uniswap_2_vs_uniswap_4', 'uniswap_2_vs_uniswap_5', 'uniswap_3_vs_pancake_1', 'uniswap_3_vs_pancake_2', 'uniswap_3_vs_uniswap_4', 'uniswap_3_vs_uniswap_5', 'uniswap_4_vs_pancake_1', 'uniswap_4_vs_pancake_2', 'uniswap_4_vs_uniswap_5', 'uniswap_5_vs_pancake_1', 'uniswap_5_vs_pancake_2']
magnitude: dropping 2 pairs: ['pancake_1_vs_pancake_2', 'uniswap_1_vs_uniswap_3']
magnitude: (77, 20) | mean log_gap: -0.3048 | pairs: 4


## Within pool-pair FE OLS

Entity-demeaned estimator (`linearmodels` PanelOLS), so the coefficient vector is just the eight
covariates. Two-way clustered (pool pair × block). In logs, each coefficient is a semi-elasticity
of the gap.

In [3]:
res = est.fit_magnitude_panel_ols(mag, two_way=True)
print(res)

                          PanelOLS Estimation Summary                           
Dep. Variable:                log_gap   R-squared:                        0.3077
Estimator:                   PanelOLS   R-squared (Between):          -6.455e+04
No. Observations:                  77   R-squared (Within):               0.3077
Date:                Thu, Aug 06 2026   R-squared (Overall):             -6598.7
Time:                        19:13:07   Log-likelihood                   -114.96
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      3.6112
Entities:                           4   P-value                           0.0016
Avg Obs:                       19.250   Distribution:                    F(8,65)
Min Obs:                       7.0000                                           
Max Obs:                       40.000   F-statistic (robust):             10.646
                            